In [35]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
import os

In [36]:
dataset = load_dataset("json", 
                       data_files={"train": "train.jsonl",
                                  "test": "test.jsonl"})

Generating train split: 4536 examples [00:00, 947572.61 examples/s]
Generating test split: 1134 examples [00:00, 866838.11 examples/s]


In [37]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 4536
    })
    test: Dataset({
        features: ['input', 'output'],
        num_rows: 1134
    })
})


In [ ]:
# adjust pre-processing functions
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

mapDict = {
    "No Hate Speech": 0,
    "Hate Speech": 1
}

def transform_labels(label):
    label = label["output"]
    result = []
    for l in label:
        result.append(mapDict[l])

    return {"label": result}

def tokenizer_function(example):
    return tokenizer(example["input"], padding=True,  truncation=True)

In [ ]:
# pre-processing the dataset
tokenized_datasets = dataset.map(tokenizer_function, batched=True)
tokenized_datasets = tokenized_datasets.map(transform_labels, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map: 100%|██████████| 1134/1134 [00:00<00:00, 313102.54 examples/s]


In [ ]:
# define training parameters
from transformers import TrainingArguments

output_dir = "./bert-hate-speech-test"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=200,
    save_total_limit=2,
    save_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
# set the environment for training
from transformers import Trainer, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased")

os.environ["WANDB_DISABLE"]="true"
os.environ["WANDB_MODE"]="offline"

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6947.33it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [ ]:
# adjust the acc metric
import numpy as np 
import evaluate

metric = evaluate.load("accuracy")

def compute_metric(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis = -1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
# define training method
trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metric
)

In [ ]:
# trainning the model
trainer.train()

/Users/alysson/Desktop/Rocketseat/ia-para-devs/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Accuracy
200,0.624758,0.613996,0.671958
400,0.623406,0.616162,0.675485
600,0.616164,0.615091,0.678131
800,0.646039,0.632307,0.678131
1000,0.646716,0.628033,0.678131
1200,0.629974,0.629002,0.678131
1400,0.647636,0.628037,0.678131
1600,0.635045,0.628498,0.678131
1701,0.614227,0.628754,0.678131


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.27it/s]
/Users/alysson/Desktop/Rocketseat/ia-para-devs/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.69it/s]
/Users/alysson/Desktop/Rocketseat/ia-para-devs/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.39it/s]
/Users/alysson/Desktop/Rocketseat/ia-para-devs/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model sh

TrainOutput(global_step=1701, training_loss=0.6262845299231312, metrics={'train_runtime': 947.0629, 'train_samples_per_second': 14.369, 'train_steps_per_second': 1.796, 'total_flos': 506657337087840.0, 'train_loss': 0.6262845299231312, 'epoch': 3.0})

In [ ]:
# evaluate model 
trainer.evaluate()

/Users/alysson/Desktop/Rocketseat/ia-para-devs/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Accuracy
0.614227,0.614836,1701,0.678131


{'eval_loss': 0.6148356795310974, 'eval_accuracy': 0.6781305114638448}

In [ ]:
# save model weights
trainer.save_model()

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.43it/s]


In [ ]:
import torch

model_path = "./bert-hate-speech-test"

# device detection
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Dispositivo: {device}")

# load tokenizer and trained model
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.to(device)
model.eval()

# mapping the classes
id2label = {
    0: "No Hate Speech",
    1: "Hate Speech"
}

# text for test
text = "Como a cultura popular é algo lindo"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

# move the tensors from the device
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

probs = torch.softmax(outputs.logits, dim=-1)
pred = torch.argmax(probs, dim=-1).item()

print(f"Texto: {text}")
print(f"Classe: {id2label[pred]}")
print(f"Confiança: {probs[0][pred]:.2%}")
print(f"Probabilidades: {probs.squeeze().cpu().numpy()}")

Dispositivo: mps


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6851.43it/s]


Texto: Como a cultura popular é algo lindo
Classe: No Hate Speech
Confiança: 53.89%
Probabilidades: [0.5388591 0.4611408]
